In [ ]:
#IMPORTS AND INITIAL TOOL FUNCTIONS


import sys
import os

from weather_tools import get_weather, get_weather_schema
from typing import Any, cast
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import MessageParam
from anthropic.types import Message
from anthropic.types import ToolParam
from typing import Any, cast

import json

#Using same helper functions as the other notebooks for consistency and ease of use
#sys.path.insert(0, os.path.join(os.path.dirname(__file__) if '__file__' in dir() else os.getcwd(), '..'))
#from helper_functions import add_user_message, add_assistant_message, chat, DEFAULT_MODEL, client


# Tools and Schemas
from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
})

set_reminder_schema = ToolParam({
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
})

batch_tool_schema = ToolParam({
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
})

pass


# TOOL:  CURRENT DATE TIME FUNCTION

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format or date_format.strip() == "":
        raise ValueError("date_format cannot be empty.")
    return datetime.now().strftime(date_format)
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the provided date_format string. The date_format parameter should follow Python's strftime format codes, allowing for flexible output formats. For example, using '%Y-%m-%d %H:%M:%S' will return the current date and time in the format '2026-03-29 14:30:00'.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "The format string for the output date and time, following Python's strftime format codes. Defaults to '%Y-%m-%d %H:%M:%S'.",
            }
        },
        "required": [],
    },
})



In [ ]:
#Helper Functions

load_dotenv()
client = Anthropic()
DEFAULT_MODEL = "claude-sonnet-4-0"


def add_user_message(messages: list[MessageParam], message) -> None:
    user_message = cast(MessageParam, {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    })
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], message) -> None:
    assistant_message = cast(MessageParam, {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    })
    messages.append(assistant_message)


def chat(
    messages: list[MessageParam],
    max_tokens: int = 1000,
    model: str = DEFAULT_MODEL,
    temperature: float = 0.6,
    system: str | None = None,
    stop_sequences: list[str] | None = None,
    tools: list[ToolParam] | None = None,
) -> Message:
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    if tools is not None:
        params["tools"] = tools
    
    message = client.messages.create(**params)
    return message

def text_from_message (message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )
    

def run_tool (tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "get_weather":
        return get_weather(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "add_duration_to_datetime":     
        return add_duration_to_datetime(**tool_input)
    else:
        raise ValueError(f"Tool {tool_name} not found")
    
    
def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []
    
    for tool_request in tool_requests:
        try:
            tool_output= run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type":"tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error":False
            }
            tool_result_blocks.append(tool_result_block)
        except Exception as e:
            tool_result_block = {
                "type":"tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error {e}",
                "is_error":True
            }
            tool_result_blocks.append(tool_result_block)
            
    return tool_result_blocks
    
def run_conversation(messages):
    while True:
        response = chat(messages, tools=
                        [get_current_datetime_schema, 
                         get_weather_schema, 
                         set_reminder_schema, 
                         add_duration_to_datetime_schema])
        add_assistant_message(messages, response)
        
        print(text_from_message(response))
        
        if response.stop_reason!= "tool_use":
            break
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

In [ ]:
# CALL DATE TIME FUNCTION

import json

# CALLING get curent datetime tool 

messages = []
messages.append({
    "role": "user",
    "content": "what is the exact time formatted as HH:MM:SS?"
})
client = Anthropic()
response = client.messages.create(
    model=DEFAULT_MODEL,
    tools=[get_current_datetime_schema],
    messages=messages,
    max_tokens=500
)

# Tool call usage may change the way Claude sends back the response.
messages.append({
    "role": "assistant",
    "content": response.content,
})

print(json.dumps(messages, indent=2, ensure_ascii=False, default=str))

In [ ]:
 #TEST CURRENT DATE TIME
response.content[0]

result = get_current_datetime(**response.content[0].input)

In [ ]:
# Add tool result block

messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": result,
            "is_error": False
            
        }
    ]
})

messages

In [ ]:
response = client.messages.create(
    model=DEFAULT_MODEL,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],

)
response

In [ ]:
# Now running run conversation  function

messages = []
add_user_message(messages, "what is the exact time formatted as HH:MM:SS? Also what is the current time in SS fomrat?")
run_conversation(messages)

In [164]:
# RUNNING MULTIPLE TOOL CALLS
messages = []
add_user_message(messages, 
                 "Set a reminder for my doctors appointment, Its 20 days from today")
run_conversation(messages)
print (json.dumps(messages, indent=2, ensure_ascii=False, default=str))

I'll help you set a reminder for your doctor's appointment in 20 days. Let me first get the current date and then calculate the date 20 days from now.


----
Setting the following reminder for 2026-04-21T00:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on Tuesday, April 21, 2026. The reminder will notify you at the beginning of that day (midnight) so you don't forget about your appointment.

If you'd like to adjust the time of the reminder (for example, to remind you the day before or at a specific time), just let me know!
[
  {
    "role": "user",
    "content": "Set a reminder for my doctors appointment, Its 20 days from today"
  },
  {
    "role": "assistant",
    "content": [
      "TextBlock(citations=None, text=\"I'll help you set a reminder for your doctor's appointment in 20 days. Let me first get the current date and then calculate the date 20 days from now.\", type='text')",
      "ToolUseBlock(id='toolu_01TodY1uRXhvS3ea5WJ5H5F9',